# اليوم الخامس عشر — تطبيقات Streamlit

تحويل تحليل بيانات القروض إلى لوحة تفاعلية، وتجربة نشر التطبيق، وإنشاء تطبيق لحساب الوزن على الكواكب.
رابط تطبيق لوحة القروض:
https://loan-dashboard-zehjq9qjcztwncdmrvky8b.streamlit.app/

## Task 1 — إعداد أول تطبيق

In [1]:
!pip install -q streamlit

In [2]:
%%writefile hello.py
import streamlit as st

st.title("أول تطبيق لي باستخدام Streamlit")
st.write("أنا متدربة في دبلوم الذكاء الاصطناعي، وأتعلم بناء تطبيقات تفاعلية لعرض البيانات.")
st.write("عدد مهام اليوم:", 4)

Writing hello.py


In [3]:
import subprocess
import time
from google.colab import output

hello_process = subprocess.Popen(
    [
        "streamlit", "run", "hello.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=open("hello.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
output.serve_kernel_port_as_iframe(8501, height=500)

<IPython.core.display.Javascript object>

In [4]:
print(open("hello.log").read())



2026-11-17 08:53:08.136 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.73.106.62:8501

2026-11-17 08:53:15.150 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c6-1wcwdqtfm4boz-c.us-east1.prod.r.colab.dev, host=m-s-use1c6-1wcwdqtfm4boz.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 08:53:17.025 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c6-1wcwdqtfm4boz-c.us-east1.prod.r.colab.dev, host=m-s-use1c6-1wcwdqtfm4boz.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 08:53:18.596 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c6-1wcwdqtfm4boz-c.us-east1.prod.r.colab.dev, host=m-s-use1c6-1wcwdqtfm4boz.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 08:5

In [5]:
import subprocess
import time
from urllib.parse import urlsplit
from google.colab import output

# إيقاف التشغيل السابق
if hello_process.poll() is None:
    hello_process.terminate()
    hello_process.wait(timeout=10)

# معرفة عنوان العرض في كولاب
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
parts = urlsplit(proxy_url)
colab_origin = f"{parts.scheme}://{parts.netloc}"

# إعادة التشغيل مع السماح بعنوان كولاب
hello_process = subprocess.Popen(
    [
        "streamlit", "run", "hello.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.corsAllowedOrigins", colab_origin
    ],
    stdout=open("hello.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
output.serve_kernel_port_as_iframe(8501, height=500)

<IPython.core.display.Javascript object>

In [6]:
from google.colab import files
uploaded = files.upload()

Saving loan_clean.csv to loan_clean.csv


In [7]:
import pandas as pd

df = pd.read_csv("loan_clean.csv")

print("عدد الصفوف والأعمدة:", df.shape)
print("\nأسماء الأعمدة:")
print(df.columns.tolist())

display(df.head())

عدد الصفوف والأعمدة: (614, 13)

أسماء الأعمدة:
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1,Urban,1
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1,Rural,0
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1,Urban,1
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1,Urban,1
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1,Urban,1


## Task 2 — لوحة تحكم القروض
عرض مؤشرات القروض ورسوماتها مع فلاتر تفاعلية للمنطقة والتعليم والدخل.

In [8]:
%%writefile app.py
from pathlib import Path

import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

st.set_page_config(
    page_title="Loan Dashboard",
    page_icon="📊",
    layout="wide"
)

@st.cache_data
def load_data():
    path = Path(__file__).parent / "loan_clean.csv"
    return pd.read_csv(path)

df = load_data()

st.title("📊 لوحة تحكم القروض")
st.write(
    "استكشاف طلبات القروض ومؤشرات الموافقة "
    "حسب المنطقة والمستوى التعليمي والدخل."
)

st.sidebar.header("فلاتر البيانات")

regions = st.sidebar.multiselect(
    "المنطقة",
    options=sorted(df["Property_Area"].dropna().unique()),
    default=sorted(df["Property_Area"].dropna().unique())
)

education = st.sidebar.radio(
    "المستوى التعليمي",
    ["الكل"] + sorted(df["Education"].dropna().unique())
)

income_min = int(df["ApplicantIncome"].min())
income_max = int(df["ApplicantIncome"].max())

if income_min < income_max:
    max_income = st.sidebar.slider(
        "الحد الأعلى لدخل المتقدم",
        min_value=income_min,
        max_value=income_max,
        value=income_max
    )
else:
    max_income = income_max
    st.sidebar.info("جميع المتقدمين لديهم الدخل نفسه.")

filtered = df[
    df["Property_Area"].isin(regions)
    & (df["ApplicantIncome"] <= max_income)
].copy()

if education != "الكل":
    filtered = filtered[filtered["Education"] == education]

if filtered.empty:
    st.warning("لا توجد طلبات تطابق الفلاتر. جرّبي تعديل الاختيارات.")
    st.stop()

total = len(filtered)
approval_rate = filtered["Loan_Status"].mean() * 100
median_loan = filtered["LoanAmount"].median()

m1, m2, m3 = st.columns(3)

m1.metric("عدد الطلبات", f"{total:,}")
m2.metric("نسبة الموافقة", f"{approval_rate:.1f}%")
m3.metric("وسيط مبلغ القرض", f"{median_loan:,.1f}")

st.caption(
    "مبالغ القروض معروضة بوحدة البيانات الأصلية. "
    "حالة القرض: 1 = موافقة، 0 = رفض."
)

st.subheader("الرسومات التفاعلية")

numeric_columns = filtered.select_dtypes(include="number").columns.tolist()

selected_column = st.selectbox(
    "اختاري عمودًا رقميًا لعرض توزيعه",
    numeric_columns,
    index=numeric_columns.index("ApplicantIncome")
)

sns.set_theme(style="whitegrid")
left, right = st.columns(2)

with left:
    st.write("**حالة القروض حسب المنطقة**")

    fig1, ax1 = plt.subplots(figsize=(7, 4))

    sns.countplot(
        data=filtered,
        x="Property_Area",
        hue="Loan_Status",
        palette={0: "#E59177", 1: "#43A6AC"},
        ax=ax1
    )

    ax1.set_xlabel("Property area")
    ax1.set_ylabel("Number of applications")
    ax1.legend(title="Status: 0=Rejected, 1=Approved")

    fig1.tight_layout()
    st.pyplot(fig1)
    plt.close(fig1)

with right:
    st.write(f"**توزيع {selected_column}**")

    values = filtered[selected_column].dropna()
    fig2, ax2 = plt.subplots(figsize=(7, 4))

    if values.empty:
        ax2.text(
            0.5, 0.5, "No numeric values",
            ha="center", va="center",
            transform=ax2.transAxes
        )
    else:
        sns.histplot(
            x=values,
            bins=20,
            color="#43A6AC",
            ax=ax2
        )

    ax2.set_xlabel(selected_column)
    ax2.set_ylabel("Frequency")

    fig2.tight_layout()
    st.pyplot(fig2)
    plt.close(fig2)

with st.expander("عرض البيانات بعد تطبيق الفلاتر"):
    st.dataframe(filtered, use_container_width=True)

Writing app.py


In [9]:
import subprocess
import time
from urllib.parse import urlsplit
from google.colab import output

proxy_url = output.eval_js("google.colab.kernel.proxyPort(8502)")
parts = urlsplit(proxy_url)
dashboard_origin = f"{parts.scheme}://{parts.netloc}"

if "dashboard_process" in globals():
    if dashboard_process.poll() is None:
        dashboard_process.terminate()
        dashboard_process.wait(timeout=10)

dashboard_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port=8502",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.corsAllowedOrigins", dashboard_origin
    ],
    stdout=open("dashboard.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
output.serve_kernel_port_as_iframe(8502, height=850)

<IPython.core.display.Javascript object>

In [10]:
%%writefile requirements.txt
streamlit
pandas
seaborn
matplotlib

Writing requirements.txt


In [11]:
from zipfile import ZipFile
from google.colab import files

with ZipFile("loan-dashboard.zip", "w") as archive:
    for filename in ["app.py", "loan_clean.csv", "requirements.txt"]:
        archive.write(filename)

files.download("loan-dashboard.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 3 — نشر التطبيق

رفعت ملفات app.py وloan_clean.csv وrequirements.txt إلى مستودع loan-dashboard في GitHub، ثم نشرت التطبيق على Streamlit Community Cloud باستخدام الفرع main وملف التشغيل app.py.

رابط التطبيق:
https://loan-dashboard-zehjq9qjcztwncdmrvky8b.streamlit.app/

فتحت التطبيق من الجوال للتأكد من ظهوره. وعند تجربة فلتر Graduate على الموقع المنشور، تغير عدد الطلبات إلى 480، ونسبة الموافقة إلى 70.8%، ووسيط مبلغ القرض إلى 130.

### سجل أخطاء النشر

لم تظهر رسالة خطأ أثناء النشر.

إذا نسيت إضافة مكتبة مطلوبة إلى requirements.txt ولم تكن موجودة في بيئة التشغيل، فقد يظهر خطأ ModuleNotFoundError. والحل هو إضافة المكتبة إلى الملف وتحديثه في GitHub ثم إعادة تشغيل التطبيق.

## Task 4 — ميزان الفضاء

إنشاء تطبيق تفاعلي لحساب قراءة الوزن على الكواكب والقمر، ومقارنتها بالوزن على الأرض، وعرض رسم بياني لجميع العوالم.

In [12]:
%%writefile space_scale.py
import streamlit as st
import pandas as pd

st.set_page_config(page_title="ميزان الفضاء", page_icon="🚀")

GRAVITY = {
    "Mercury": 0.38,
    "Venus": 0.90,
    "Earth": 1.00,
    "Moon": 0.17,
    "Mars": 0.38,
    "Jupiter": 2.53,
    "Saturn": 1.07,
    "Uranus": 0.89,
    "Neptune": 1.14
}

st.title("🚀 ميزان الفضاء")
st.write("اكتشفي كيف تتغير قراءة وزنك باختلاف الجاذبية.")

earth_weight = st.number_input(
    "وزنك على الأرض بالكيلوجرام",
    min_value=1.0,
    value=70.0,
    step=1.0
)

planet = st.selectbox("اختاري العالم", list(GRAVITY))

factor = GRAVITY[planet]
space_weight = earth_weight * factor
difference = space_weight - earth_weight

st.metric(
    f"قراءة الوزن على {planet}",
    f"{space_weight:.2f} كجم",
    delta=f"{difference:+.2f} كجم مقارنة بالأرض",
    delta_color="off"
)

if factor < 1:
    st.success(
        f"ستكونين أخف! نظريًا يمكنك القفز أعلى "
        f"بنحو {1 / factor:.2f} مرة، بافتراض سرعة الانطلاق نفسها."
    )
elif factor > 1:
    st.warning("ستكونين أثقل بسبب زيادة الجاذبية!")
else:
    st.info("هذه جاذبية الأرض، لذلك قراءة وزنك لا تتغير.")

st.subheader("مقارنة الوزن على جميع العوالم")

comparison = pd.DataFrame({
    "World": list(GRAVITY),
    "Weight": [earth_weight * g for g in GRAVITY.values()]
}).set_index("World")

st.bar_chart(comparison)

st.caption(
    "الحساب يستخدم معاملات الجاذبية المعطاة في المهمة. "
    "الكتلة ثابتة، والمعروض قراءة وزن مكافئة بالكيلوجرام."
)

Writing space_scale.py


In [13]:
import subprocess
import time
from urllib.parse import urlsplit
from google.colab import output

proxy_url = output.eval_js("google.colab.kernel.proxyPort(8503)")
parts = urlsplit(proxy_url)
space_origin = f"{parts.scheme}://{parts.netloc}"

if "space_process" in globals():
    if space_process.poll() is None:
        space_process.terminate()
        space_process.wait(timeout=10)

space_process = subprocess.Popen(
    [
        "streamlit", "run", "space_scale.py",
        "--server.port=8503",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.corsAllowedOrigins", space_origin
    ],
    stdout=open("space.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
output.serve_kernel_port_as_iframe(8503, height=850)

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download("space_scale.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>